# LocalRetro 环境配置教程

## 概述

LocalRetro 是一种基于**局部反应模板**的逆合成预测方法，发表于 JACS 2021。

> **论文**: *Retrosynthesis Prediction with Local Template Retrieval* (Chen & Jung, 2021, Chemical Science)  
> **核心思想**: 与传统的全局模板匹配方法不同，LocalRetro 将反应模板分解为**原子级别**和**键级别**的局部模板，使用 MPNN + 全局反应性注意力机制 (Global Reactivity Attention) 为产物分子中的每个原子/键预测其对应的局部编辑操作。

### 与 GLN 的主要区别

| 特性 | GLN | LocalRetro |
|------|-----|------------|
| 模板类型 | 全局反应模板（整个反应） | **局部模板**（原子/键级别编辑） |
| 预测方式 | 分层次预测：中心→模板→验证 | **直接预测每个原子/键的编辑操作** |
| GNN 框架 | 自定义 MeanField/GGNN | **MPNN + 全局注意力** |
| 图框架 | torch-geometric | **DGL + DGLLife** |
| 优势 | 可解释性强 | 模板数量少、泛化性好 |

### 本教程包含三份 notebook

| 序号 | Notebook | 内容 |
|------|----------|------|
| 1 | **环境配置**（本文件） | 安装依赖、验证环境 |
| 2 | 数据处理 | 最小化的 LocalRetro 数据处理流程 |
| 3 | 模型展示 | LocalRetro 推理原理与模型计算架构 |

---

## 1. 核心依赖一览

LocalRetro 原始代码依赖较老的版本。本教程适配到当前主流版本：

| 依赖库 | 原始版本 | 教程版本（推荐） | 说明 |
|--------|----------|-----------------|------|
| Python | 3.7 | **3.10+** | 现代化 Python |
| PyTorch | 1.0+ | **2.0+** | 新版 PyTorch |
| RDKit | 2019+ | **2023.09+** | 化学信息学工具包 |
| DGL | 0.5.2+ | **1.1+** | 深度图学习框架 |
| DGLLife | 0.2.6+ | **0.3+** | DGL 生命科学扩展 |
| numpy | 1.16+ | **1.24+** | 数值计算 |
| pandas | - | **2.0+** | 数据分析 |

> **与 GLN 的关键区别**: LocalRetro 使用 **DGL (Deep Graph Library)** 而非 torch-geometric 作为图神经网络框架，特征化使用 DGLLife 的内置原子/键特征化器 (`WeaveAtomFeaturizer`, `CanonicalBondFeaturizer`)。

## 2. 创建虚拟环境并安装依赖

以下代码会在项目的 `envs/` 目录下创建一个独立的 Python 虚拟环境，避免与系统环境冲突。

In [1]:
import os
from pathlib import Path

# ====== 定位项目根目录 ======
def find_project_root(start=None):
    """向上查找项目根目录（包含 teaching_demos/ 的目录）"""
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / 'teaching_demos').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

PROJECT_ROOT = find_project_root()
ENV_DIR = PROJECT_ROOT / 'envs' / 'localretro_tutorial_envs'
SOURCE_REPO = PROJECT_ROOT / 'source_repos' / 'LocalRetro'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'虚拟环境目录: {ENV_DIR}')
print(f'LocalRetro 源码目录: {SOURCE_REPO}')

项目根目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
虚拟环境目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localretro_tutorial_envs
LocalRetro 源码目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/LocalRetro


In [4]:
%%bash -s "$ENV_DIR"

ENV_DIR=$1

# 让 bash 能识别 conda 命令
source "$(conda info --base)/etc/profile.d/conda.sh"

# 如果虚拟环境不存在则创建
if [ ! -d "$ENV_DIR" ]; then
    echo "==> 正在创建 Conda 环境 $ENV_DIR ..."
    conda create -y -p "$ENV_DIR" python=3.11
    echo "==> Conda 环境创建完成"
else
    echo "==> Conda 环境已存在: $ENV_DIR"
fi

# 激活环境
conda activate "$ENV_DIR"

echo "==> 安装核心依赖 ..."
conda install -y -c conda-forge \
    numpy \
    scipy \
    pandas \
    tqdm \
    rdkit \
    matplotlib \
    ipykernel \
    scikit-learn

echo "==> 安装 PyTorch ..."
pip install -q torch==2.4.0 torchvision --index-url https://download.pytorch.org/whl/cpu

echo "==> 安装 DGL (图学习框架) ..."
pip install dgl -f https://data.dgl.ai/wheels/torch-2.4/repo.html
pip install dgllife
pip install jupyter notebook
echo "==> 所有依赖安装完成"

==> Conda 环境已存在: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localretro_tutorial_envs
==> 安装核心依赖 ...
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: ...working... done




==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda





# All requested packages already installed.

==> 安装 PyTorch ...
==> 安装 DGL (图学习框架) ...
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
Looking in links: https://data.dgl.ai/wheels/torch-2.4/repo.html
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
==> 所有依赖安装完成


### 将虚拟环境注册为 Jupyter Kernel（可选）

如果你在 Jupyter 环境中运行，可以将新环境注册为独立的 kernel：

In [5]:
%%bash -s "$ENV_DIR"
source "$1/bin/activate"
python -m ipykernel install --user --name localretro_tutorial --display-name "LocalRetro Tutorial"
echo "==> Kernel 'LocalRetro Tutorial' 注册完成，请在 Jupyter 中选择该 kernel"

bash: 行 1: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localretro_tutorial_envs/bin/activate: 没有那个文件或目录


Installed kernelspec localretro_tutorial in /home/xiaoruiwang/.local/share/jupyter/kernels/localretro_tutorial
==> Kernel 'LocalRetro Tutorial' 注册完成，请在 Jupyter 中选择该 kernel


## 3. 验证环境

运行以下代码验证所有核心依赖是否安装成功。

> **提示**: 确保当前 notebook 使用的是 `localretro_tutorial` kernel 或对应虚拟环境中的 Python。

In [1]:
import sys
print(f'Python 版本: {sys.version}')
print(f'Python 路径: {sys.executable}')
print()

Python 版本: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
Python 路径: /home/xiaoruiwang/data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/envs/localretro_tutorial_envs/bin/python



In [2]:
# ====== 逐一检查依赖 ======

checks = {}

# 1. PyTorch
try:
    import torch
    checks['PyTorch'] = f'{torch.__version__} (CUDA: {torch.cuda.is_available()})'
except ImportError as e:
    checks['PyTorch'] = f'FAIL: {e}'

# 2. RDKit
try:
    from rdkit import Chem, rdBase
    checks['RDKit'] = rdBase.rdkitVersion
except ImportError as e:
    checks['RDKit'] = f'FAIL: {e}'

# 3. NumPy
try:
    import numpy as np
    checks['NumPy'] = np.__version__
except ImportError as e:
    checks['NumPy'] = f'FAIL: {e}'

# 4. DGL
try:
    import dgl
    checks['DGL'] = dgl.__version__
except ImportError as e:
    checks['DGL'] = f'FAIL: {e}'

# 5. DGLLife
try:
    import dgllife
    checks['DGLLife'] = dgllife.__version__
except ImportError as e:
    checks['DGLLife'] = f'FAIL: {e}'

# 6. pandas
try:
    import pandas as pd
    checks['pandas'] = pd.__version__
except ImportError as e:
    checks['pandas'] = f'FAIL: {e}'

# 7. scikit-learn
try:
    import sklearn
    checks['scikit-learn'] = sklearn.__version__
except ImportError as e:
    checks['scikit-learn'] = f'FAIL: {e}'

# 8. matplotlib
try:
    import matplotlib
    checks['matplotlib'] = matplotlib.__version__
except ImportError as e:
    checks['matplotlib'] = f'FAIL: {e}'

# 9. tqdm
try:
    import tqdm
    checks['tqdm'] = tqdm.__version__
except ImportError as e:
    checks['tqdm'] = f'FAIL: {e}'

print('='*50)
print('环境检查结果')
print('='*50)
all_ok = True
for name, status in checks.items():
    flag = '✓' if 'FAIL' not in str(status) else '✗'
    if 'FAIL' in str(status):
        all_ok = False
    print(f'  {flag} {name:20s} {status}')

print('='*50)
if all_ok:
    print('所有依赖检查通过！')
else:
    print('部分依赖缺失，请检查上方安装步骤。')

环境检查结果
  ✓ PyTorch              2.4.0+cu121 (CUDA: True)
  ✓ RDKit                2025.09.6
  ✓ NumPy                2.4.2
  ✓ DGL                  2.4.0
  ✓ DGLLife              0.3.2
  ✓ pandas               3.0.1
  ✓ scikit-learn         1.8.0
  ✓ matplotlib           3.10.8
  ✓ tqdm                 4.67.3
所有依赖检查通过！


## 4. 验证 RDKit 基础功能

LocalRetro 的核心操作依赖 RDKit 的以下功能：
- **SMILES 解析与标准化**: 将分子表示统一为 canonical 形式
- **反应 SMARTS 应用**: 用模板对产物执行逆合成变换
- **原子映射 (Atom Mapping)**: 追踪原子在反应前后的对应关系
- **子结构修改**: 应用局部模板后修正 H 数、电荷、手性等

In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

# ====== 4.1 SMILES 标准化 ======
# LocalRetro 在数据预处理中会去除原子映射编号

def canonicalize_smiles(smiles):
    """将 SMILES 标准化：去原子映射编号"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles
    [atom.SetAtomMapNum(0) for atom in mol.GetAtoms()]
    return Chem.MolToSmiles(mol)

test_smiles = [
    '[CH3:1][OH:2]',           # 带原子映射的甲醇
    'c1ccccc1',                 # 苯
    'CC(=O)Oc1ccccc1',         # 乙酸苯酯
]

print('SMILES 标准化示例（去除原子映射）：')
print('-' * 50)
for sm in test_smiles:
    cano = canonicalize_smiles(sm)
    print(f'  原始: {sm:30s} → 标准化: {cano}')

SMILES 标准化示例（去除原子映射）：
--------------------------------------------------
  原始: [CH3:1][OH:2]                  → 标准化: CO
  原始: c1ccccc1                       → 标准化: c1ccccc1
  原始: CC(=O)Oc1ccccc1                → 标准化: CC(=O)Oc1ccccc1


In [4]:
# ====== 4.2 反应 SMARTS 应用 ======
# LocalRetro 的核心是将局部模板 (reaction SMARTS) 应用到产物上，生成反应物

from rdkit.Chem import rdChemReactions

# 示例：酰胺键断裂的局部模板
# 产物中 N-C(=O) → 反应物中 NH2 + C(=O)Cl
template_smarts = '([N:1]-[C:2](=[O:3]))>>([N:1].[Cl:99][C:2](=[O:3]))'
product_smiles = 'CC(=O)Nc1ccccc1'  # 乙酰苯胺

rxn = rdChemReactions.ReactionFromSmarts(template_smarts)
prod_mol = Chem.MolFromSmiles(product_smiles)

# 给产物分子的原子设置映射编号（LocalRetro 用原子 idx 做映射）
[atom.SetAtomMapNum(atom.GetIdx()) for atom in prod_mol.GetAtoms()]

outcomes = rxn.RunReactants([prod_mol])

print(f'模板 SMARTS: {template_smarts}')
print(f'产物 SMILES: {product_smiles}')
print(f'模板应用结果 ({len(outcomes)} 种可能):')
for i, products in enumerate(outcomes):
    reactant_smiles = '.'.join([Chem.MolToSmiles(p) for p in products])
    print(f'  结果 {i+1}: {reactant_smiles}')

模板 SMARTS: ([N:1]-[C:2](=[O:3]))>>([N:1].[Cl:99][C:2](=[O:3]))
产物 SMILES: CC(=O)Nc1ccccc1
模板应用结果 (1 种可能):
  结果 1: CC(=O)Cl.N[c:4]1[cH:5][cH:6][cH:7][cH:8][cH:9]1


In [5]:
# ====== 4.3 原子/键特征提取（DGLLife 方式） ======
# LocalRetro 使用 DGLLife 内置的 WeaveAtomFeaturizer 和 CanonicalBondFeaturizer
# 这与 GLN 手工定义特征的方式完全不同

from dgllife.utils import WeaveAtomFeaturizer, CanonicalBondFeaturizer, smiles_to_bigraph
from functools import partial

# LocalRetro 使用的原子类型列表（64 种元素）
atom_types = ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe',
         'As', 'Al', 'I', 'B', 'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti',
         'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb',
         'W', 'Ru', 'Nb', 'Re', 'Te', 'Rh', 'Ta', 'Tc', 'Ba', 'Bi', 'Hf', 'Mo', 'U', 'Sm', 'Os', 'Ir',
         'Ce', 'Gd', 'Ga', 'Cs']

node_featurizer = WeaveAtomFeaturizer(atom_types=atom_types)
edge_featurizer = CanonicalBondFeaturizer(self_loop=True)

# 用 DGLLife 工具将 SMILES 转换为 DGL 图
smiles = 'CC(=O)Oc1ccccc1'  # 乙酸苯酯
g = smiles_to_bigraph(smiles, 
                      node_featurizer=node_featurizer,
                      edge_featurizer=edge_featurizer,
                      canonical_atom_order=False,
                      add_self_loop=True)

print(f'分子: {smiles}')
print(f'DGL 图结构: {g}')
print(f'  节点数（原子数）: {g.num_nodes()}')
print(f'  边数（含自环）: {g.num_edges()}')
print(f'  节点特征维度: {g.ndata["h"].shape}  (WeaveAtomFeaturizer)')
print(f'  边特征维度:   {g.edata["e"].shape}  (CanonicalBondFeaturizer + self_loop)')
print()
print('节点特征 (前3个原子):')
mol = Chem.MolFromSmiles(smiles)
for i in range(min(3, g.num_nodes())):
    atom = mol.GetAtomWithIdx(i)
    print(f'  原子 {i} ({atom.GetSymbol()}): 特征向量维度={g.ndata["h"][i].shape[0]}, '
          f'非零元素数={int(g.ndata["h"][i].sum().item())}')

分子: CC(=O)Oc1ccccc1
DGL 图结构: Graph(num_nodes=10, num_edges=30,
      ndata_schemes={'h': Scheme(shape=(80,), dtype=torch.float32)}
      edata_schemes={'e': Scheme(shape=(13,), dtype=torch.float32)})
  节点数（原子数）: 10
  边数（含自环）: 30
  节点特征维度: torch.Size([10, 80])  (WeaveAtomFeaturizer)
  边特征维度:   torch.Size([30, 13])  (CanonicalBondFeaturizer + self_loop)

节点特征 (前3个原子):
  原子 0 (C): 特征向量维度=80, 非零元素数=2
  原子 1 (C): 特征向量维度=80, 非零元素数=2
  原子 2 (O): 特征向量维度=80, 非零元素数=2


## 5. 验证 DGL 图操作

LocalRetro 使用 DGL 来构建和操作分子图。与 GLN 使用的 torch-geometric 不同，DGL 的图操作接口有自己的特点。

In [6]:
import torch
import dgl

# ====== DGL 图操作示例 ======
# LocalRetro 在前向传播中需要以下核心 DGL 操作：
# 1. dgl.batch / dgl.unbatch：批量处理多个图
# 2. remove_self_loop：去除自环（self-loop）
# 3. 节点/边特征操作

# 构建两个简单的分子图
# 图1: 丙烷 C-C-C (3个节点)
g1 = dgl.graph(([0, 1, 1, 2], [1, 0, 2, 1]))
g1.ndata['h'] = torch.randn(3, 4)  # 3个节点，4维特征

# 图2: 乙烷 C-C (2个节点)
g2 = dgl.graph(([0, 1], [1, 0]))
g2.ndata['h'] = torch.randn(2, 4)  # 2个节点，4维特征

# batch 操作：将多个图合并为一个大图
bg = dgl.batch([g1, g2])
print(f'批量图: 节点数={bg.num_nodes()}, 边数={bg.num_edges()}')
print(f'批量图节点特征形状: {bg.ndata["h"].shape}')

# unbatch 操作：将大图拆分回单个图
graphs = dgl.unbatch(bg)
print(f'\n拆分后: {len(graphs)} 个图')
for i, g in enumerate(graphs):
    print(f'  图 {i}: 节点数={g.num_nodes()}, 边数={g.num_edges()}')

批量图: 节点数=5, 边数=6
批量图节点特征形状: torch.Size([5, 4])

拆分后: 2 个图
  图 0: 节点数=3, 边数=4
  图 1: 节点数=2, 边数=2


In [7]:
from dgllife.model import MPNNGNN

# ====== MPNN 消息传递示例 ======
# LocalRetro 使用 DGLLife 的 MPNNGNN 作为分子编码器

# 简单的 MPNN 层
mpnn = MPNNGNN(
    node_in_feats=4,        # 节点输入特征维度
    node_out_feats=8,       # 节点输出特征维度
    edge_in_feats=3,        # 边输入特征维度
    edge_hidden_feats=16,   # 边隐藏层维度
    num_step_message_passing=3  # 消息传递步数
)

# 构建测试图（加自环以匹配 LocalRetro 的图构建方式）
g = dgl.graph(([0, 1, 1, 2, 0, 1, 2], [1, 0, 2, 1, 0, 1, 2]))
node_feats = torch.randn(3, 4)
edge_feats = torch.randn(g.num_edges(), 3)

# 前向传播
out = mpnn(g, node_feats, edge_feats)

print(f'MPNN 输入: 节点特征 {node_feats.shape}, 边特征 {edge_feats.shape}')
print(f'MPNN 输出: {out.shape}')
print(f'MPNN 参数量: {sum(p.numel() for p in mpnn.parameters()):,}')
print('\n✓ DGL + DGLLife MPNN 消息传递机制正常工作！')

MPNN 输入: 节点特征 torch.Size([3, 4]), 边特征 torch.Size([7, 3])
MPNN 输出: torch.Size([3, 8])
MPNN 参数量: 1,632

✓ DGL + DGLLife MPNN 消息传递机制正常工作！


## 6. LocalRetro 源代码结构概览

为了方便后续教程的理解，这里展示 LocalRetro 的源代码目录结构及各模块功能。

In [8]:
structure = """
LocalRetro 源代码结构
══════════════════════════════════════════════════════════════
LocalRetro/
├── LocalTemplate/                # 局部模板工具
│   ├── template_extractor.py     # 模板提取器（从rdChiral修改）
│   ├── template_extract_utils.py # 模板提取辅助函数
│   └── template_decoder.py       # 模板解码器（预测→反应物SMILES）
│
├── preprocessing/                # 数据预处理
│   ├── Extract_from_train_data.py # 从训练集提取模板
│   └── Run_preprocessing.py       # 数据标注与预处理
│
├── scripts/                      # 训练/测试/推理
│   ├── models.py                 # LocalRetro_model 定义
│   ├── model_utils.py            # 注意力机制、特征操作
│   ├── dataset.py                # USPTODataset 数据集类
│   ├── utils.py                  # 模型加载、数据加载工具
│   ├── Train.py                  # 训练脚本
│   ├── Test.py                   # 测试脚本
│   ├── get_edit.py               # 预测结果提取
│   └── Decode_predictions.py     # 解码预测为SMILES
│
├── data/
│   ├── configs/                  # 模型超参数配置
│   │   └── default_config.json   # 默认配置
│   ├── USPTO_50K/               # USPTO-50K 数据集
│   │   ├── atom_templates.csv    # 124个原子级别模板
│   │   ├── bond_templates.csv    # 548个键级别模板  
│   │   └── template_infos.csv    # 658个模板详细信息
│   └── USPTO_MIT/               # USPTO-MIT 数据集
│
├── models/
│   └── LocalRetro_USPTO_50K.pth  # 预训练模型
│
└── Retrosynthesis.py             # 端到端推理脚本
"""
print(structure)


LocalRetro 源代码结构
══════════════════════════════════════════════════════════════
LocalRetro/
├── LocalTemplate/                # 局部模板工具
│   ├── template_extractor.py     # 模板提取器（从rdChiral修改）
│   ├── template_extract_utils.py # 模板提取辅助函数
│   └── template_decoder.py       # 模板解码器（预测→反应物SMILES）
│
├── preprocessing/                # 数据预处理
│   ├── Extract_from_train_data.py # 从训练集提取模板
│   └── Run_preprocessing.py       # 数据标注与预处理
│
├── scripts/                      # 训练/测试/推理
│   ├── models.py                 # LocalRetro_model 定义
│   ├── model_utils.py            # 注意力机制、特征操作
│   ├── dataset.py                # USPTODataset 数据集类
│   ├── utils.py                  # 模型加载、数据加载工具
│   ├── Train.py                  # 训练脚本
│   ├── Test.py                   # 测试脚本
│   ├── get_edit.py               # 预测结果提取
│   └── Decode_predictions.py     # 解码预测为SMILES
│
├── data/
│   ├── configs/                  # 模型超参数配置
│   │   └── default_config.json   # 默认配置
│   ├── USPTO_50K/               # USPTO-50K 数据集
│

## 7. 快速理解 LocalRetro 的工作流程

LocalRetro 的整体流程可以总结为以下管线：

```
┌───────────────────────────────────────────────────────────────┐
│                        数据预处理阶段                          │
│                                                               │
│  原始反应数据 (SMILES with atom mapping)                       │
│    ↓                                                          │
│  提取局部模板 → 分为原子模板 + 键模板                           │
│    ↓                                                          │
│  为每条反应标注编辑标签 (原子编辑位点 + 键编辑位点)              │
│    ↓                                                          │
│  生成 DGL 图 + 标签文件                                       │
└───────────────────────────────────────────────────────────────┘

┌───────────────────────────────────────────────────────────────┐
│                        训练阶段                                │
│                                                               │
│  MPNN 编码产物分子 → 原子嵌入                                  │
│    ↓                                                          │
│  原子特征拼接 → 键特征                                         │
│    ↓                                                          │
│  Global Reactivity Attention → 全局上下文增强                  │
│    ↓                                                          │
│  分类头：原子→原子模板类别, 键→键模板类别                       │
│    ↓                                                          │
│  CrossEntropy 损失 → 反向传播                                  │
└───────────────────────────────────────────────────────────────┘

┌───────────────────────────────────────────────────────────────┐
│                        推理阶段                                │
│                                                               │
│  输入：产物 SMILES                                             │
│    ↓                                                          │
│  (1) MPNN + Attention → 预测每个原子/键的编辑概率               │
│    ↓                                                          │
│  (2) Top-K 选择：合并原子编辑和键编辑，取概率最高的 K 个         │
│    ↓                                                          │
│  (3) 模板应用：用局部模板对产物做反应 → 生成反应物               │
│    ↓                                                          │
│  (4) 后处理：修正 H 数、电荷、手性 → 输出反应物 SMILES          │
└───────────────────────────────────────────────────────────────┘
```

### 核心公式

LocalRetro 将逆合成预测建模为一个**局部分类问题**：

对于产物分子 $G = (V, E)$ 中的每个原子 $v_i \in V$ 和每条键 $e_{ij} \in E$：

$$P(t_a | v_i, G) = \text{Softmax}(\text{MLP}_a(\mathbf{h}_i^{\text{att}}))$$

$$P(t_b | e_{ij}, G) = \text{Softmax}(\text{MLP}_b(\mathbf{h}_{ij}^{\text{att}}))$$

其中：
- $t_a$ 是原子级别模板（124 种 + 1 个无操作类别）
- $t_b$ 是键级别模板（548 种 + 1 个无操作类别）
- $\mathbf{h}_i^{\text{att}}$ 是经过全局注意力增强后的原子特征
- $\mathbf{h}_{ij}^{\text{att}}$ 是经过全局注意力增强后的键特征

最终预测通过比较所有原子和键的编辑概率，选取 Top-K 进行模板应用和解码。

## 8. 总结

本 notebook 完成了以下工作：

1. ✅ 安装了 LocalRetro 教程所需的所有 Python 依赖
2. ✅ 验证了 RDKit、PyTorch、DGL、DGLLife 等核心库
3. ✅ 演示了 LocalRetro 用到的 RDKit 核心操作（SMILES标准化、反应SMARTS应用）
4. ✅ 演示了 LocalRetro 用到的 DGL 图操作（batch/unbatch、MPNN消息传递）
5. ✅ 概览了 LocalRetro 的代码结构和工作流程

**下一步** → 请打开 `2_数据处理.ipynb`，学习 LocalRetro 的完整数据处理流程。